<table style="width:100%">
  <tr>
    <td valign="top"><img src="../data/img/FER_logo_2.png" width=300 height=80 align="left"></td>
    <td valign="top"><img src="../data/img/LARES_2_transparent.png" width=250 height=80 align="right"></td>
  </tr>
 </table>

# Unsupervised learning

## Setup

Run this cell first, in every notebook. It fetches the course repository into
the Colab session and moves into the `notebooks/` folder, so that the
`../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory.
Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, we are set.

In [ ]:
# --- SETUP: run this first ---
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/unizg-fer-lares/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## Import modules

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

In [ ]:
import matplotlib.pyplot as plt

tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": False,
    "font.family": "serif",
    # Use 26pt font in plots
    "axes.labelsize": 16,
    "font.size": 16,
    "figure.titlesize": 16,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 14,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14
}

plt.rcParams.update(tex_fonts)

#%pylab inline
%matplotlib inline
# %matplotlib notebook
%config InlineBackend.figure_format='svg'
plt.rcParams['figure.figsize'] = (10, 7)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Auxiliary functions

In [ ]:
# aux function for elbow method implementation
def elbow_method(X_train, n):
    wcss = []
    # calculate Within Cluster Sum of Squares (WCSS) for KMeans with 1-n clusters
    for i in range(1, n):
        kmeans = KMeans(n_clusters = i, init = 'k-means++', random_state = 42)
        kmeans.fit(X_train)
        wcss.append(kmeans.inertia_)
    
    fig = plt.figure(figsize = (10, 10))
    
    plt.plot(range(1, n), wcss)
    plt.title('The Elbow Method')
    plt.xlabel('Number of clusters')
    plt.ylabel('WCSS')
    plt.show()

In [ ]:
# aux function for plotting data with characterized clusters
def plot_clusters(X_train, y_kmeans, feat_1, feat_2):
    fig = plt.figure(figsize = (8, 8))
    for i, cluster_center in enumerate(model_kmeans.cluster_centers_):
        plt.scatter(X_train[y_kmeans == i, 0], X_train[y_kmeans == i, 1], s = 200, label = 'Cluster '+str(i+1))

    plt.scatter(model_kmeans.cluster_centers_[:, 0], model_kmeans.cluster_centers_[:, 1], s = 200, c = 'k', marker='x', label = 'Centroids')

    plt.title('Clusters')
    plt.xlabel(feat_1)
    plt.ylabel(feat_2)
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
# aux function for plotting PCA features
def plot_pca_component(pca_features, component_num):
    first_comp = pca_features[:, component_num]
    comps = pd.DataFrame(list(zip(first_comp, X_scaled.columns)), columns=["weights", "features"])
    comps["abs_weights"] = comps["weights"].apply(lambda x: np.abs(x))
    ax = sns.barplot(data=comps.sort_values("abs_weights", ascending=False).head(10),
                     x="weights",y="features",palette="Blues_d")
    ax.set_title("PCA Component Makeup: #" + str(component_num))
    plt.show()

## Unsupervised learning
0. Preprocess data

1. K-means clustering

2. Hierarchical clustering

3. Principal component analysis

4. HANDS-ON: clustering with reduced dimensionality


## 0. (Pre)process data

Let's start with a simple presentation example and consider a dataset of mall customers and their characteristics: spending score and annual income.

Dataset available at: https://www.kaggle.com/shwetabh123/mall-customers

In [ ]:
dataset = pd.read_csv('../data/mall_customers/mall_customers.csv')

In [ ]:
dataset.head()

In [ ]:
# choose only 'Annual Income (k$)' and 'Spending Score (1-100)' columns
X_train = dataset.loc[:, ['Annual Income (k$)', 'Spending Score (1-100)']].values
# simple scatter plot to visualize the data distribution
fig = plt.figure(figsize = (8, 8))
plt.scatter(X_train[:,0],X_train[:,1], s = 200)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.grid(True)
plt.show()

## 1. K-means clustering

### Use the elbow method to find the optimal number of clusters

In [ ]:
# elbow method up to 10 clusters
elbow_method(X_train, 10)

### Training the K-Means model on the dataset - based on the elbow method results

In [ ]:
# train K-Means model with 5 clusters (elbow method)
model_kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42)
y_kmeans = model_kmeans.fit_predict(X_train)

### Visualize the clusters

In [ ]:
# plot the characterized clusters
plot_clusters(X_train, y_kmeans, 'Annual Income (k$)', 'Spending Score (1-100)')

## 2. Hierarchical clustering

In [ ]:
"""
Perform hierarchical clustering on samples using the
linkage() function with the method='complete' keyword argument.
Assign the result to mergings.
"""
from scipy.cluster.hierarchy import linkage, dendrogram

mergings = linkage(X_train, method='complete')

In [ ]:
"""
Plot a dendrogram using the dendrogram() function on mergings,
specifying the keyword arguments labels=varieties, leaf_rotation=90,
and leaf_font_size=6.
"""
fig, ax = plt.subplots(figsize = (12,6))

dendrogram(mergings,
           ax=ax,
           leaf_rotation=90,
           leaf_font_size=6,
           )

ax.set_title('Hierarchical Clustering Dendrogram')
ax.set_xlabel('sample index')
ax.set_ylabel('distance')

# ax.set_xlim([0, 1500])

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

dendrogram(
    mergings,
    truncate_mode='lastp',  # show only the last p merged clusters
    p=12,  # show only the last p merged clusters
    show_leaf_counts=True,  # otherwise numbers in brackets are counts
    leaf_rotation=90.,
    leaf_font_size=12.,
    show_contracted=True,  # to get a distribution impression in truncated branches
)

ax.set_title('Hierarchical Clustering Dendrogram (truncated)')
ax.set_xlabel('sample index')
ax.set_ylabel('distance')

plt.show()

**Exercise:** do the same with other types of linkages (single, complete, average, weighted, centroid, median, ward)
Reference: https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html

## 3. Principal component analysis

Firstly, the data: let's use Seeds dataset (UCI ML): https://archive.ics.uci.edu/dataset/236/seeds

The data consists of seven geometric parameters of wheat kernels were measured:
1. area A,
2. perimeter P,
3. compactness C = 4*pi*A/P^2,
4. length of kernel,
5. width of kernel,
6. asymmetry coefficient
7. length of kernel groove.

All of these parameters were real-valued continuous.

In [ ]:
# load the data
seeds_df = pd.read_csv("../data/wheat/seeds_dataset.txt", sep="\t", header=None)
seeds_df.rename(columns={0: 'area', 1: 'perimeter', 2: 'compactness', 3: 'kernel_length',
                         4: 'kernel_width', 5: 'asymmetry', 6: 'groove_lenght', 7: 'grain_type'}, inplace=True)
seeds_df

#### Correlation

In [ ]:
seeds_corr = seeds_df.iloc[:, :-1].corr()

mask = np.zeros_like(seeds_corr, dtype=bool)
mask[np.tril_indices_from(mask)] = True

sns.heatmap(seeds_corr, mask=~mask, cmap='Wistia', annot=True)
plt.title('Heatmap for the Data')
plt.show();

#### Separate input features and corresponding outputs

In [ ]:
y = seeds_df['grain_type']
X = seeds_df.drop('grain_type', axis=1)

### Scaling

In [ ]:
scaler = MinMaxScaler()

X_scaled = pd.DataFrame(scaler.fit_transform(X))
X_scaled.columns = X.columns
X_scaled.index = seeds_df.index

In [ ]:
X_scaled.describe()

## PCA

In [ ]:
# Create a PCA instance, fit the data, and transform it
pca = PCA().fit(X_scaled)
pca_features = pca.transform(X_scaled)
pca_features

### Plot variance for PCA components

In [ ]:
# Plot the explained variances
features = range(pca.n_components_)
plt.bar(features, pca.explained_variance_ratio_)
plt.xlabel('PCA feature')
plt.ylabel('variance')
plt.xticks(features)
plt.show()

In [ ]:
plot_pca_component(pca_features=pca_features, component_num=0)

In [ ]:
plot_pca_component(pca_features=pca_features, component_num=1)

## 4. HANDS-ON: clustering with reduced dimensionality

The wheat data is transformed using PCA.
Cluster this transformed data and compare with true values (y).

In [ ]:
# define the data - use 2 most influential features
X = ???

In [ ]:
# train the model, random_state=42
model = ???
y_hat = ???

In [ ]:
# compare with true data
df = pd.DataFrame({
    "true": y,
    "predicted": y_hat,
})
df

In [ ]:
# find out which cluster represent which grain type
for c in range(3):
    for g in range(1, 4):
        N = (df.true[df.predicted==c] == g).sum()
        print(f"Cluster {c} correspond to {N} samples of grain type {g}")

In [ ]:
# remap clusters to grain types
def remap(c):
    if c==0:
        return 2
    elif c==1:
        return 3
    elif c==2:
        return 1
    else:
        raise ValueError("Cannot be anything aprt from 0, 1, and 2")

df2 = df.copy()
df2.predicted = df2.predicted.apply(remap)

In [ ]:
# get metrics
from sklearn.metrics import classification_report

print(classification_report(df2.true, df2.predicted))

In [ ]:
# plot the clusters, their centroids, and the errors
centroids = model.cluster_centers_
errors = (df2.predicted != df2.true).values

fig = plt.figure(figsize=(12, 8))

# samples colored by the predicted grain type
for g in range(1, 4):
    mask = (df2.predicted == g).values
    plt.scatter(X[mask, 0], X[mask, 1], s=80, label='Predicted grain type ' + str(g))

# centroids, annotated with the grain type they were remapped to
plt.scatter(centroids[:, 0], centroids[:, 1], s=300, c='k', marker='x',
            linewidths=3, label='Centroids')
# for c, centroid in enumerate(centroids):
#     plt.annotate('Grain type ' + str(remap(c)), centroid, textcoords='offset points',
#                  xytext=(12, 10), fontsize=13, fontweight='bold')

# circle the samples where the predicted grain type is wrong
plt.scatter(X[errors, 0], X[errors, 1], s=250, facecolors='none', edgecolors='r',
            linewidths=2, label='Errors (' + str(errors.sum()) + ')')

plt.title('Clusters of the PCA-transformed wheat data')
plt.xlabel('PCA component 1')
plt.ylabel('PCA component 2')
plt.grid(True)
plt.legend()
plt.show()



_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - Foundations of AI_  \
_Notebook: 3_Unsupervised_learning_

_References:_
- _Scikit-learn documentation, https://scikit-learn.org/stable/index.html_
- _SciPy documentation, https://docs.scipy.org/doc/scipy/index.html_

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_

